# Cu FEFF encoder study

This replaces the old v1/v2/depth-ablation notebooks with one smaller workflow.

Promising combinations trained here:

| group | feature / encoder | members |
|---|---|---:|
| learned | attention encoder 96D, seeds 42/43 | 2 encoders × 3 heads |
| learned | attention encoder 128D, seeds 42/43 | 2 encoders × 3 heads |
| learned local pooling | learned `site_weighted5`, 96D/128D, seeds 42/43 | 4 feature sets × 3 heads |
| learned local pooling | learned `site_shells_3_5_weighted`, 96D/128D, seeds 42/43 | 4 feature sets × 3 heads |
| frozen M3GNet | `site_weighted5` local feature | 3 heads |
| frozen M3GNet | `site_shells_3_5_weighted` local feature | 3 heads |

The learned-local rows use the trained encoder's node embeddings with the same pooling formulas as the frozen depth-ablation features.
The final table reports member means and ensemble eta, plus paper ExpertXAS and old v1 ensemble references.


In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import re
import shutil

import dgl
import lightning.pytorch as pl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from matgl import load_model
from matgl.config import DEFAULT_ELEMENTS
from matgl.ext.pymatgen import Structure2Graph
try:
    from matgl.graph.compute import compute_pair_vector_and_distance, compute_theta_and_phi, create_line_graph
except Exception:
    from matgl.graph._compute_dgl import compute_pair_vector_and_distance, compute_theta_and_phi, create_line_graph
from matgl.layers import (
    MLP as M3GNetMLP,
    ActivationFunction,
    BondExpansion,
    EmbeddingBlock,
    GatedMLP,
    M3GNetBlock,
    SphericalBesselWithHarmonics,
    ThreeBodyInteractions,
)
from matgl.utils.cutoff import polynomial_cutoff
from pymatgen.core import Lattice, Structure
from torch import nn
from torch.utils.data import DataLoader, Dataset

import matgl.layers._basis as matgl_basis
import matgl.layers._three_body as matgl_three_body
import matgl.utils.maths as matgl_math

from omnixas.data.ml_data import MLData, MLSplits
from omnixas.featurizer.m3gnet_featurizer import M3GNetFeaturizer
from omnixas.model.xasblock import XASBlock
from omnixas.model.xasblock_regressor import XASBlockRegressor

REPO_ROOT = Path.cwd().resolve()
while not ((REPO_ROOT / "pyproject.toml").exists() and (REPO_ROOT / "omnixas").exists()):
    REPO_ROOT = REPO_ROOT.parent

TASK = "Cu_FEFF"
DATA_DIR = REPO_ROOT / "tutorial_omnixas" / "ml_data"
ID_DIR = REPO_ROOT / "tutorial_omnixas" / "material_id_and_site"
RAW_ROOT = Path(os.environ.get("OMNIXAS_DATA_ROOT", REPO_ROOT.parent / "OmniXAS_data")) / "materialscloud_omnixas_raw" / "extracted"
OUT_ROOT = REPO_ROOT / "output" / "training" / "cuFeffEncoderStudy"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
if not RAW_ROOT.exists():
    raise FileNotFoundError(f"Missing raw OmniXAS data: {RAW_ROOT}")

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision("medium")

SPLITS = ["train", "val", "test"]
y_true = {s: np.loadtxt(DATA_DIR / f"{TASK}_{s}_y.txt", dtype=np.float32) for s in SPLITS}

FEATURE_SCALE = 1000.0
BATCH_SIZE = 32
NUM_WORKERS = 4
CUTOFF = 4.0
N_BLOCKS = 3
GNN_DROPOUT = 0.1
ENCODER_WIDTHS = [96, 128]
ENCODER_SEEDS = [42, 43]
ENCODER_EPOCHS = 150
ENCODER_LR = 1e-3

HEAD_DIMS = [600, 600, 400]
HEAD_EPOCHS = 400
HEAD_CONFIGS = [
    {"seed": 42, "dropout": 0.50, "lr": 1e-3},
    {"seed": 44, "dropout": 0.30, "lr": 1e-3},
    {"seed": 45, "dropout": 0.25, "lr": 7e-4},
]

LEARNED_POOLINGS = ["attention", "site_weighted5", "site_shells_3_5_weighted"]
FROZEN_FEATURES = {
    "frozen_site_weighted5": 384,
    "frozen_site_shells_3_5_weighted": 576,
}

REFERENCE = pd.DataFrame([
    {"feature": "paper ExpertXAS", "kind": "reference", "n_members": 1, "ensemble_val_eta": np.nan, "ensemble_test_eta": 5.19},
    {"feature": "old v1 20-member ensemble", "kind": "reference", "n_members": 20, "ensemble_val_eta": 7.48, "ensemble_test_eta": 7.64},
])

TRAIN_LEARNED_ENCODERS = True
BUILD_FROZEN_FEATURES = True
TRAIN_HEADS = True

In [ ]:
def eta(pred, target, train_y=y_true["train"]):
    med = np.median(np.mean((target - pred) ** 2, axis=1))
    base = np.median(np.mean((target - train_y.mean(axis=0, keepdims=True)) ** 2, axis=1))
    return float(base / med)


def torch_load(path: Path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def patch_matgl_gpu_constants() -> None:
    def _call_sbf(self, r):
        cutoff = torch.as_tensor(self.cutoff, dtype=r.dtype, device=r.device)
        roots = matgl_basis.SPHERICAL_BESSEL_ROOTS[: self.max_l, : self.max_n].to(r.device, dtype=r.dtype)
        factor = torch.sqrt(torch.as_tensor(2.0, dtype=r.dtype, device=r.device) / cutoff**3)
        r = r.clamp(max=cutoff)
        return torch.cat([
            self.funcs[i](r[:, None] * roots[i][None, :] / cutoff) * factor / torch.abs(self.funcs[i + 1](roots[i][None, :]))
            for i in range(self.max_l)
        ], dim=1)

    def _combine(sbf, shf, max_n: int, max_l: int, use_phi: bool):
        if sbf.size(0) == 0:
            return sbf
        if use_phi:
            repeats = torch.repeat_interleave(2 * torch.arange(max_l, device=sbf.device) + 1, max_n)
            block_size = 2 * torch.arange(max_l, device=sbf.device) + 1
        else:
            repeats = torch.ones(max_l * max_n, dtype=torch.long, device=sbf.device)
            block_size = [1] * max_l
        cols = torch.arange(shf.size(1), device=shf.device)
        idx, start = [], 0
        for block in block_size:
            block = int(block.item()) if torch.is_tensor(block) else int(block)
            idx.append(torch.tile(cols[start : start + block], [max_n]))
            start += block
        sbf = torch.repeat_interleave(sbf, repeats, 1)
        return torch.reshape(sbf * torch.index_select(shf, 1, torch.cat(idx)), [-1, max_n * max_l * (max_l if use_phi else 1)])

    def _scatter_sum(x, segment_ids, num_segments: int, dim: int):
        segment_ids = matgl_math.broadcast(segment_ids.to(x.device), x, dim)
        size = list(x.size())
        size[dim] = 0 if segment_ids.numel() == 0 else num_segments
        return torch.zeros(size, dtype=x.dtype, device=x.device).scatter_add_(dim, segment_ids, x)

    matgl_basis.SphericalBesselFunction._call_sbf = _call_sbf
    matgl_basis.combine_sbf_shf = _combine
    matgl_three_body.combine_sbf_shf = _combine
    matgl_math.scatter_sum = _scatter_sum
    matgl_three_body.scatter_sum = _scatter_sum


def parse_feff_structure(path: Path) -> Structure:
    abc = angles = None
    species, coords = [], []
    site_re = re.compile(r"^\*\s+\d+\s+([A-Z][a-z]?)\s+([-+0-9.eE]+)\s+([-+0-9.eE]+)\s+([-+0-9.eE]+)")
    for line in path.read_text(errors="ignore").splitlines():
        if line.startswith("TITLE abc:"):
            abc = [float(x) for x in line.split(":", 1)[1].split()[:3]]
        elif line.startswith("TITLE angles:"):
            angles = [float(x) for x in line.split(":", 1)[1].split()[:3]]
        elif match := site_re.match(line):
            species.append(match.group(1))
            coords.append([float(match.group(i)) for i in range(2, 5)])
    if abc is None or angles is None or not species:
        raise ValueError(f"Could not parse FEFF structure: {path}")
    return Structure(Lattice.from_parameters(*abc, *angles), species, coords, coords_are_cartesian=False)


def load_structure(material_id: str, site: int) -> Structure:
    material_dir = RAW_ROOT / "FEFF" / "Cu" / material_id
    poscar = material_dir / "POSCAR"
    if poscar.exists():
        return Structure.from_file(poscar)
    return parse_feff_structure(material_dir / "FEFF-XANES" / f"{site:03d}_Cu" / "feff.inp")


class FEFFDataset(Dataset):
    def __init__(self, split: str):
        ids = [line.strip().rsplit("_", 1) for line in (ID_DIR / f"{TASK}_{split}.txt").read_text().splitlines() if line.strip()]
        y = np.atleast_2d(np.loadtxt(DATA_DIR / f"{TASK}_{split}_y.txt", dtype=np.float32))
        self.rows = [(mid, int(site), torch.as_tensor(yi, dtype=torch.float32)) for (mid, site), yi in zip(ids, y, strict=True)]
        self.cache = {}

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        mid, site, y = self.rows[idx]
        if mid not in self.cache:
            self.cache[mid] = load_structure(mid, site)
        return self.cache[mid], site, y


class CollateGraphs:
    def __init__(self, encoder):
        self.converter = Structure2Graph(encoder.element_types, encoder.cutoff)

    def graph(self, structure: Structure):
        out = self.converter.get_graph(structure)
        if len(out) == 2:
            graph, _ = out
            lat = torch.as_tensor(structure.lattice.matrix, dtype=torch.float32)
        else:
            graph, lat, _ = out
            lat = torch.as_tensor(lat[0] if getattr(lat, "ndim", 0) == 3 else lat, dtype=torch.float32)
        if "pbc_offshift" in graph.edata:
            graph.edata["pbc_offshift"] = graph.edata["pbc_offshift"].to(torch.float32)
        else:
            graph.edata["pbc_offshift"] = graph.edata["pbc_offset"].to(torch.float32) @ lat
        if "pos" in graph.ndata:
            graph.ndata["pos"] = graph.ndata["pos"].to(torch.float32)
        elif "frac_coords" in graph.ndata:
            graph.ndata["pos"] = graph.ndata["frac_coords"].to(torch.float32) @ lat
        else:
            graph.ndata["pos"] = torch.as_tensor(structure.cart_coords, dtype=torch.float32)
        graph.edata["bond_vec"], graph.edata["bond_dist"] = compute_pair_vector_and_distance(graph)
        return graph

    def __call__(self, batch):
        graphs, sites, y = [], [], []
        offset = 0
        for structure, site, yi in batch:
            graph = self.graph(structure)
            graphs.append(graph)
            sites.append(offset + site)
            y.append(yi)
            offset += graph.num_nodes()
        return {"graph": dgl.batch(graphs), "site": torch.tensor(sites), "y": torch.stack(y).float()}


patch_matgl_gpu_constants()

In [ ]:
class AttentionReadout(nn.Module):
    def __init__(self, dim, cutoff):
        super().__init__()
        self.dim, self.cutoff = dim, cutoff
        self.q = nn.Linear(dim, dim, bias=False)
        self.k = nn.Linear(dim, dim, bias=False)
        self.v = nn.Linear(dim, dim, bias=False)

    def forward(self, g, node, site):
        src, dst, eid = g.in_edges(site, form="all")
        row = torch.zeros(g.num_nodes(), dtype=torch.long, device=node.device)
        row[site] = torch.arange(len(site), device=node.device)
        row = row[dst]
        dist = g.edata["bond_dist"][eid]
        logits = (self.q(node[dst]) * self.k(node[src])).sum(-1) / self.dim**0.5
        logits = logits + torch.log(polynomial_cutoff(dist, self.cutoff) + 1e-9)
        peak = torch.full((len(site),), -torch.inf, device=node.device).scatter_reduce(0, row, logits, "amax", include_self=False)
        w = torch.exp(logits - peak[row])
        norm = torch.zeros(len(site), device=node.device).scatter_add(0, row, w).clamp_min(1e-12)
        ctx = torch.zeros(len(site), self.dim, device=node.device).index_add(0, row, (w / norm[row]).unsqueeze(-1) * self.v(node[src]))
        cn = torch.zeros(len(site), device=node.device).scatter_add(0, row, torch.ones_like(dist))
        cn3 = torch.zeros(len(site), device=node.device).scatter_add(0, row, (dist <= 3.0).float())
        dmin = torch.full((len(site),), self.cutoff, device=node.device).scatter_reduce(0, row, dist, "amin")
        dmean = torch.zeros(len(site), device=node.device).scatter_add(0, row, dist) / cn.clamp_min(1)
        return torch.cat([node[site], ctx, torch.stack([cn / 10, cn3 / 10, dmin / self.cutoff, dmean / self.cutoff], dim=-1)], dim=-1)


class XASEncoder(nn.Module):
    def __init__(self, dim):
        super().__init__()
        act = ActivationFunction["swish"].value()
        degree = 9
        self.dim = dim
        self.element_types = DEFAULT_ELEMENTS
        self.cutoff = CUTOFF
        self.threebody_cutoff = CUTOFF
        self.bond_expansion = BondExpansion(3, 3, CUTOFF)
        self.basis_expansion = SphericalBesselWithHarmonics(3, 3, CUTOFF, use_smooth=False, use_phi=False)
        self.embedding = EmbeddingBlock(degree_rbf=degree, dim_node_embedding=dim, dim_edge_embedding=dim, ntypes_node=len(DEFAULT_ELEMENTS), activation=act)
        self.three_body_interactions = nn.ModuleList([
            ThreeBodyInteractions(
                update_network_atom=M3GNetMLP(dims=[dim, degree], activation=nn.Sigmoid(), activate_last=True),
                update_network_bond=GatedMLP(in_feats=degree, dims=[dim], use_bias=False),
            ) for _ in range(N_BLOCKS)
        ])
        self.graph_layers = nn.ModuleList([
            M3GNetBlock(degree=degree, activation=act, conv_hiddens=[dim, dim], dim_node_feats=dim, dim_edge_feats=dim, dropout=GNN_DROPOUT)
            for _ in range(N_BLOCKS)
        ])
        self.readout = AttentionReadout(dim, CUTOFF)
        self.out_dim = 2 * dim + 4

    def node_features(self, g):
        g.edata["rbf"] = self.bond_expansion(g.edata["bond_dist"])
        lg = create_line_graph(g.to("cpu"), self.threebody_cutoff).to(g.device)
        lg.apply_edges(compute_theta_and_phi)
        basis = self.basis_expansion(lg)
        cutoff = polynomial_cutoff(g.edata["bond_dist"], self.threebody_cutoff)
        node, edge, state = self.embedding(g.ndata["node_type"], g.edata["rbf"], None)
        for three_body, block in zip(self.three_body_interactions, self.graph_layers):
            edge = three_body(g, lg, basis, cutoff, node, edge)
            edge, node, state = block(g, edge, node, state)
        return node

    def pooled_features(self, g, site):
        node = self.node_features(g)
        src, dst, eid = g.in_edges(site, form="all")
        row = torch.zeros(g.num_nodes(), dtype=torch.long, device=node.device)
        row[site] = torch.arange(len(site), device=node.device)
        row = row[dst]
        dist = g.edata["bond_dist"][eid]

        def weighted(mask):
            out = torch.zeros(len(site), self.dim, device=node.device)
            if not mask.any():
                return out
            r, s = row[mask], src[mask]
            w = 1.0 / dist[mask].clamp_min(1e-6)
            norm = torch.zeros(len(site), device=node.device).scatter_add(0, r, w).clamp_min(1e-12)
            return out.index_add(0, r, (w / norm[r]).unsqueeze(-1) * node[s])

        return {
            "attention": self.readout(g, node, site),
            "site_weighted5": torch.cat([node[site], weighted(dist <= 5.0)], dim=-1),
            "site_shells_3_5_weighted": torch.cat([node[site], weighted(dist <= 3.0), weighted((dist > 3.0) & (dist <= 5.0))], dim=-1),
        }

    def forward(self, g, site):
        return self.pooled_features(g, site)["attention"]


class LitEncoder(pl.LightningModule):
    def __init__(self, dim):
        super().__init__()
        self.encoder = XASEncoder(dim)
        self.head = nn.Sequential(
            nn.Linear(self.encoder.out_dim, 128), nn.BatchNorm1d(128), nn.SiLU(), nn.Dropout(0.25),
            nn.Linear(128, 128), nn.BatchNorm1d(128), nn.SiLU(), nn.Dropout(0.25),
            nn.Linear(128, 141), nn.Softplus(),
        )
        self.val_mses = []

    def step(self, batch, split):
        pred = self.head(self.encoder(batch["graph"].to(self.device), batch["site"].to(self.device)) * FEATURE_SCALE)
        y = batch["y"].to(self.device)
        loss = ((pred - y) ** 2).mean()
        self.log(f"{split}_loss", loss, on_epoch=True, prog_bar=True)
        if split == "val":
            self.val_mses.append(((pred - y) ** 2).mean(dim=1).detach())
        return loss

    def training_step(self, batch, _):
        return self.step(batch, "train")

    def on_validation_epoch_start(self):
        self.val_mses = []

    def validation_step(self, batch, _):
        return self.step(batch, "val")

    def on_validation_epoch_end(self):
        if self.val_mses:
            self.log("val_median_mse", torch.cat(self.val_mses).median(), prog_bar=True)

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=ENCODER_LR, weight_decay=1e-5)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, ENCODER_EPOCHS, eta_min=1e-6)
        return {"optimizer": opt, "lr_scheduler": {"scheduler": sched, "interval": "epoch"}}


In [ ]:
train_ds, val_ds = FEFFDataset("train"), FEFFDataset("val")

for dim in ENCODER_WIDTHS:
    for seed in ENCODER_SEEDS:
        run_dir = OUT_ROOT / f"learned_{dim}d_seed{seed}"
        ckpt_dir = run_dir / "encoder_ckpts"
        if (ckpt_dir / "DONE").exists() or not TRAIN_LEARNED_ENCODERS:
            print("encoder cached/skipped:", run_dir.name)
            continue
        pl.seed_everything(seed, workers=True)
        lit = LitEncoder(dim)
        collate = CollateGraphs(lit.encoder)
        trainer = pl.Trainer(
            max_epochs=ENCODER_EPOCHS,
            accelerator="auto",
            devices=1,
            callbacks=[ModelCheckpoint(ckpt_dir, filename="best-{epoch:03d}-{val_median_mse:.5f}", monitor="val_median_mse", mode="min", save_top_k=1, save_last=True)],
            logger=CSVLogger(str(run_dir), name="encoder_logs"),
            log_every_n_steps=10,
        )
        last = ckpt_dir / "last.ckpt"
        trainer.fit(
            lit,
            DataLoader(train_ds, BATCH_SIZE, shuffle=True, collate_fn=collate, num_workers=NUM_WORKERS),
            DataLoader(val_ds, BATCH_SIZE, shuffle=False, collate_fn=collate, num_workers=NUM_WORKERS),
            ckpt_path=str(last) if last.exists() else None,
        )
        (ckpt_dir / "DONE").write_text("ok")
        torch.cuda.empty_cache()

for run_dir in sorted(OUT_ROOT.glob("learned_*d_seed*")):
    ckpt = run_dir / "encoder_ckpts" / "last.ckpt"
    if not ckpt.exists():
        continue
    stamp = np.int64(ckpt.stat().st_mtime_ns)
    feat_file = run_dir / "features.npz"
    needed = [f"{name}_{split}" for name in LEARNED_POOLINGS for split in SPLITS]
    if feat_file.exists():
        cached = np.load(feat_file)
        if int(cached["ckpt_stamp"]) == int(stamp) and all(key in cached for key in needed):
            print("features cached:", run_dir.name)
            continue
    dim = int(re.search(r"learned_(\d+)d_seed", run_dir.name).group(1))
    lit = LitEncoder(dim)
    lit.load_state_dict(torch_load(ckpt)["state_dict"])
    encoder = lit.encoder.to(DEVICE).eval()
    collate = CollateGraphs(encoder)
    arrays = {}
    with torch.no_grad():
        for split in SPLITS:
            chunks = {name: [] for name in LEARNED_POOLINGS}
            loader = DataLoader(FEFFDataset(split), BATCH_SIZE, shuffle=False, collate_fn=collate, num_workers=NUM_WORKERS)
            for batch in loader:
                pooled = encoder.pooled_features(batch["graph"].to(DEVICE), batch["site"].to(DEVICE))
                for name, value in pooled.items():
                    chunks[name].append((value * FEATURE_SCALE).cpu().numpy())
            for name, values in chunks.items():
                arrays[f"{name}_{split}"] = np.concatenate(values)
    np.savez_compressed(feat_file, ckpt_stamp=stamp, **arrays)
    encoder.to("cpu")
    torch.cuda.empty_cache()
    print("wrote", run_dir.name, {k: v.shape for k, v in arrays.items()})


In [ ]:
frozen_dir = OUT_ROOT / "frozen_features"
frozen_dir.mkdir(exist_ok=True)

if BUILD_FROZEN_FEATURES:
    m3gnet = load_model(str(REPO_ROOT / "models" / "M3GNet-MP-2021.2.8-PES")).model.eval()
    featurizers = {depth: M3GNetFeaturizer(model=m3gnet, n_blocks=depth) for depth in [1, 2, 3]}
    zero = np.zeros(192, dtype=np.float32)

    for split in SPLITS:
        out_paths = {name: frozen_dir / f"{name}_{split}.npy" for name in FROZEN_FEATURES}
        if all(path.exists() for path in out_paths.values()):
            print("frozen cached:", split)
            continue
        rows = {name: [] for name in FROZEN_FEATURES}
        cache = {}
        ids = [line.strip().rsplit("_", 1) for line in (ID_DIR / f"{TASK}_{split}.txt").read_text().splitlines() if line.strip()]
        for n, (mid, site_text) in enumerate(ids, 1):
            site = int(site_text)
            if mid not in cache:
                structure = load_structure(mid, site)
                node_by_depth = {depth: featurizers[depth].featurize(structure) * FEATURE_SCALE for depth in [1, 2, 3]}
                cache[mid] = structure, np.concatenate([node_by_depth[1], node_by_depth[2], node_by_depth[3]], axis=1)
            structure, node_concat = cache[mid]
            neighbors = [(nbr.index, float(nbr.nn_distance)) for nbr in structure.get_neighbors(structure[site], 6.0) if float(nbr.nn_distance) > 1e-8]
            ids_n = np.array([idx for idx, _ in neighbors], dtype=int)
            dist = np.array([d for _, d in neighbors], dtype=np.float32)
            values = node_concat[ids_n] if len(ids_n) else np.empty((0, 192), dtype=np.float32)
            mask3 = dist <= 3.0
            mask5 = dist <= 5.0
            shell3_5 = (dist > 3.0) & (dist <= 5.0)
            weighted5 = np.average(values[mask5], axis=0, weights=1 / np.maximum(dist[mask5], 1e-6)) if mask5.any() else zero
            weighted3 = np.average(values[mask3], axis=0, weights=1 / np.maximum(dist[mask3], 1e-6)) if mask3.any() else zero
            weighted3_5 = np.average(values[shell3_5], axis=0, weights=1 / np.maximum(dist[shell3_5], 1e-6)) if shell3_5.any() else zero
            rows["frozen_site_weighted5"].append(np.concatenate([node_concat[site], weighted5]))
            rows["frozen_site_shells_3_5_weighted"].append(np.concatenate([node_concat[site], weighted3, weighted3_5]))
            if n % 500 == 0:
                print(split, n, "/", len(ids))
        for name, values in rows.items():
            X = np.asarray(values, dtype=np.float32)
            assert X.shape[1] == FROZEN_FEATURES[name], (name, X.shape)
            np.save(out_paths[name], X)
            print("wrote", out_paths[name], X.shape)

In [ ]:
feature_jobs = []
for run_dir in sorted(OUT_ROOT.glob("learned_*d_seed*")):
    feat_file = run_dir / "features.npz"
    if not feat_file.exists():
        continue
    width = int(re.search(r"learned_(\d+)d_seed", run_dir.name).group(1))
    seed = int(run_dir.name.rsplit("seed", 1)[1])
    feats = np.load(feat_file)
    for pooling in LEARNED_POOLINGS:
        if not all(f"{pooling}_{split}" in feats for split in SPLITS):
            continue
        feature_jobs.append({
            "feature": f"learned_{pooling}_{width}d",
            "kind": "learned_encoder",
            "member_source": f"encoder_seed{seed}",
            "root": run_dir / pooling,
            "X": {split: feats[f"{pooling}_{split}"] for split in SPLITS},
        })

for name in FROZEN_FEATURES:
    if all((frozen_dir / f"{name}_{split}.npy").exists() for split in SPLITS):
        feature_jobs.append({
            "feature": name,
            "kind": "frozen_m3gnet",
            "member_source": "frozen",
            "root": OUT_ROOT / name,
            "X": {split: np.load(frozen_dir / f"{name}_{split}.npy") for split in SPLITS},
        })

print(pd.DataFrame([{k: v for k, v in job.items() if k != "X"} | {"dim": job["X"]["train"].shape[1]} for job in feature_jobs]))

for job in feature_jobs:
    splits = MLSplits(**{split: MLData(X=job["X"][split], y=y_true[split]) for split in SPLITS})
    for cfg in HEAD_CONFIGS:
        head_dir = job["root"] / "heads" / f"seed{cfg['seed']}_dropout{cfg['dropout']:g}_lr{cfg['lr']:g}"
        pred_file = head_dir / "preds.npz"
        if pred_file.exists() or not TRAIN_HEADS:
            continue
        if head_dir.exists():
            shutil.rmtree(head_dir)
        pl.seed_everything(cfg["seed"], workers=True)
        XASBlock.DROPOUT = cfg["dropout"]
        reg = XASBlockRegressor(
            directory=str(head_dir),
            input_dim=job["X"]["train"].shape[1],
            hidden_dims=HEAD_DIMS,
            output_dim=141,
            initial_lr=cfg["lr"],
            batch_size=BATCH_SIZE,
            max_epochs=HEAD_EPOCHS,
            use_early_stopping=False,
            use_lr_finder=False,
            monitor_metric="val_median_mse",
            shuffle=True,
            lr_scheduler="cosine",
            cosine_t_max=HEAD_EPOCHS,
            cosine_eta_min=1e-6,
        )
        reg.fit(splits).load("last")
        module = reg.model.model.to(DEVICE).eval()
        preds = {}
        with torch.no_grad():
            for split in SPLITS:
                X = job["X"][split]
                preds[split] = np.concatenate([
                    module(torch.tensor(X[i:i + 512], dtype=torch.float32, device=DEVICE)).cpu().numpy()
                    for i in range(0, len(X), 512)
                ])
        np.savez_compressed(pred_file, **preds)
        print(job["feature"], job["member_source"], head_dir.name, {s: round(eta(preds[s], y_true[s]), 3) for s in SPLITS})


In [ ]:
member_rows = []
pred_val, pred_test = [], []
for job in feature_jobs:
    for pred_file in sorted((job["root"] / "heads").glob("*/preds.npz")):
        preds = np.load(pred_file)
        member_rows.append({
            "feature": job["feature"],
            "kind": job["kind"],
            "member_source": job["member_source"],
            "head": pred_file.parent.name,
            **{f"{split}_eta": eta(preds[split], y_true[split]) for split in SPLITS},
        })
        pred_val.append(preds["val"])
        pred_test.append(preds["test"])

members = pd.DataFrame(member_rows)
if members.empty:
    raise FileNotFoundError("No head prediction files found; run the head-training cell first.")
pred_val = np.stack(pred_val)
pred_test = np.stack(pred_test)

summary_rows = []
for feature, group in members.groupby("feature", sort=False):
    idx = group.index.to_numpy()
    summary_rows.append({
        "feature": feature,
        "kind": group["kind"].iloc[0],
        "n_members": len(idx),
        "member_val_eta_mean": group["val_eta"].mean(),
        "member_test_eta_mean": group["test_eta"].mean(),
        "best_member_val_eta": group["val_eta"].max(),
        "best_member_test_eta": group.loc[group["val_eta"].idxmax(), "test_eta"],
        "ensemble_val_eta": eta(pred_val[idx].mean(axis=0), y_true["val"]),
        "ensemble_test_eta": eta(pred_test[idx].mean(axis=0), y_true["test"]),
    })

summary = pd.DataFrame(summary_rows)
summary["delta_vs_paper_expert"] = summary["ensemble_test_eta"] - 5.19
summary["delta_vs_v1_ensemble"] = summary["ensemble_test_eta"] - 7.64
summary = pd.concat([REFERENCE, summary], ignore_index=True, sort=False)

members.to_csv(OUT_ROOT / "members.csv", index=False)
summary.to_csv(OUT_ROOT / "summary.csv", index=False)

print("members:")
display(members.round(3))
print("summary:")
display(summary.round(3))

In [ ]:
if members.empty:
    raise FileNotFoundError("No members to plot yet; run the evaluation cell first.")

combo_masks = []
for feature in sorted(members["feature"].unique()):
    combo_masks.append((feature, members["feature"].eq(feature).to_numpy()))
combo_masks += [
    ("learned_attention_all", members["feature"].str.startswith("learned_attention_").to_numpy()),
    ("learned_weighted5_all", members["feature"].str.startswith("learned_site_weighted5_").to_numpy()),
    ("learned_shells_all", members["feature"].str.startswith("learned_site_shells_3_5_weighted_").to_numpy()),
    ("all_learned", members["kind"].eq("learned_encoder").to_numpy()),
    ("all_frozen", members["kind"].eq("frozen_m3gnet").to_numpy()),
    ("all_members", np.ones(len(members), dtype=bool)),
    ("best_val_member", np.arange(len(members)) == int(members["val_eta"].idxmax())),
]

combo_rows, combo_pred_test = [], {}
for name, mask in combo_masks:
    idx = np.flatnonzero(mask)
    if len(idx) == 0:
        continue
    pred_v = pred_val[idx].mean(axis=0)
    pred_t = pred_test[idx].mean(axis=0)
    combo_pred_test[name] = pred_t
    combo_rows.append({"combo": name, "n_members": len(idx), "val_eta": eta(pred_v, y_true["val"]), "test_eta": eta(pred_t, y_true["test"])})

combo_df = pd.DataFrame(combo_rows).sort_values("test_eta", ascending=False)
combo_df.to_csv(OUT_ROOT / "combo_summary.csv", index=False)
display(combo_df.round(3))

baseline_pred = np.repeat(y_true["train"].mean(axis=0, keepdims=True), len(y_true["test"]), axis=0)
baseline_mse = np.mean((y_true["test"] - baseline_pred) ** 2, axis=1)
combo_mse = {name: np.mean((y_true["test"] - pred) ** 2, axis=1) for name, pred in combo_pred_test.items()}
best_combo = combo_df.iloc[0]["combo"]
best_pred = combo_pred_test[best_combo]
best_mse = combo_mse[best_combo]


## Visual diagnostics

These plots compare many model combinations against the mean-spectrum base and old references.


In [ ]:
# 1) Histogram in the older notebook style: log10 per-spectrum MSE, baseline filled, models as steps.
positive = np.concatenate([baseline_mse[baseline_mse > 0], *[m[m > 0] for m in combo_mse.values()]])
bins = np.linspace(np.floor(np.log10(positive.min())), np.ceil(np.log10(positive.max())), 45)
fig, ax = plt.subplots(figsize=(10.5, 6.0), dpi=150)
ax.hist(np.log10(baseline_mse), bins=bins, density=True, alpha=0.24, color="gray", label=f"mean-spectrum base, med={np.median(baseline_mse):.2e}")
colors = plt.cm.tab20(np.linspace(0, 1, len(combo_df)))
for color, row in zip(colors, combo_df.itertuples(index=False)):
    mse = combo_mse[row.combo]
    ax.hist(np.log10(mse), bins=bins, density=True, histtype="step", linewidth=1.7, color=color, label=f"{row.combo} | eta={row.test_eta:.2f}")
    ax.axvline(np.log10(np.median(mse)), linestyle="--", linewidth=0.8, color=color, alpha=0.8)
ax.axvline(np.log10(np.median(baseline_mse)), color="gray", linestyle=":", linewidth=2)
ax.set_title("Cu FEFF test error distributions: model combinations vs base")
ax.set_xlabel(r"$\log_{10}(\mathrm{MSE\ per\ spectrum})$")
ax.set_ylabel("Density")
ax.legend(fontsize=7, ncol=2)
ax.grid(alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(OUT_ROOT / "combo_error_distributions.png", dpi=220)
plt.show()

# 2) Top combo eta bar chart with paper/v1 reference lines.
fig, ax = plt.subplots(figsize=(9.5, 4.8), dpi=150)
top = combo_df.head(10)
ax.barh(top["combo"][::-1], top["test_eta"][::-1], color="#2a78d6", alpha=0.85)
ax.axvline(5.19, color="gray", linestyle=":", linewidth=1.5, label="paper ExpertXAS")
ax.axvline(7.64, color="#1baf7a", linestyle="--", linewidth=1.5, label="old v1 ensemble")
ax.set_xlabel("test eta")
ax.set_title("Top Cu FEFF combinations")
ax.legend(fontsize=8)
ax.grid(axis="x", alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(OUT_ROOT / "combo_test_eta_bar.png", dpi=220)
plt.show()

# 3) Member val-vs-test scatter to show selection noise.
fig, ax = plt.subplots(figsize=(5.6, 4.8), dpi=150)
ax.scatter(members["val_eta"], members["test_eta"], s=45, alpha=0.8, color="#2a78d6", edgecolors="white", linewidths=0.7)
lo = members[["val_eta", "test_eta"]].min().min() - 0.2
hi = members[["val_eta", "test_eta"]].max().max() + 0.2
ax.plot([lo, hi], [lo, hi], ls=":", color="gray")
ax.set_xlabel("member val eta")
ax.set_ylabel("member test eta")
ax.set_title("Member validation/test noise")
ax.grid(alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(OUT_ROOT / "member_val_test_scatter.png", dpi=220)
plt.show()

# 4) Per-spectrum model-vs-base scatter; below diagonal means the model improves that spectrum.
improved = float(np.mean(best_mse < baseline_mse))
fig, ax = plt.subplots(figsize=(5.8, 5.2), dpi=150)
ax.scatter(baseline_mse, best_mse, s=14, alpha=0.5, color="#2a78d6", edgecolors="none")
lo = min(baseline_mse.min(), best_mse.min())
hi = max(baseline_mse.max(), best_mse.max())
ax.plot([lo, hi], [lo, hi], color="gray", linestyle=":", linewidth=1.5)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("mean-spectrum base MSE")
ax.set_ylabel(f"{best_combo} MSE")
ax.set_title(f"Best combo vs base per spectrum ({improved:.1%} improved)")
ax.grid(alpha=0.25, which="both")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(OUT_ROOT / "best_combo_vs_base_scatter.png", dpi=220)
plt.show()

# 5) Worst spectra panel for the best combo.
worst_idx = np.argsort(best_mse)[-9:][::-1]
x = np.arange(y_true["test"].shape[1])
fig, axes = plt.subplots(3, 3, figsize=(11, 7.5), dpi=150, sharex=True)
for ax, idx in zip(axes.flat, worst_idx):
    ax.plot(x, y_true["test"][idx], color="black", linewidth=1.4, label="truth")
    ax.plot(x, best_pred[idx], color="#2a78d6", linewidth=1.1, label="prediction")
    ax.fill_between(x, y_true["test"][idx], best_pred[idx], color="#2a78d6", alpha=0.18)
    ax.set_title(f"idx={idx} | MSE={best_mse[idx]:.2e}", fontsize=9)
    ax.grid(alpha=0.2)
axes[0, 0].legend(fontsize=8)
fig.suptitle(f"Worst test spectra for {best_combo}")
fig.supxlabel("energy grid index")
fig.supylabel("intensity")
fig.tight_layout()
fig.savefig(OUT_ROOT / "worst_spectra_best_combo.png", dpi=220)
plt.show()
